In [1]:
import pyspark
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, to_date

In [2]:
# Initialize Spark session and fetch the Cassandra driver
spark = SparkSession.builder \
    .appName("Amazon_Reviews_ETL") \
    .master("local[*]") \
    .config("spark.jars.packages", "com.datastax.spark:spark-cassandra-connector_2.12:3.5.0") \
    .config("spark.cassandra.connection.host", "127.0.0.1") \
    .config("spark.cassandra.connection.port", "9042") \
    .getOrCreate()

print("✅ Spark Session successfully created!")

:: loading settings :: url = jar:file:/Users/macpro/Documents/MyProject/venv/lib/python3.12/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /Users/macpro/.ivy2/cache
The jars for the packages stored in: /Users/macpro/.ivy2/jars
com.datastax.spark#spark-cassandra-connector_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-cbf868d4-2b35-4f75-9807-3a9ca13aef73;1.0
	confs: [default]
	found com.datastax.spark#spark-cassandra-connector_2.12;3.5.0 in central
	found com.datastax.spark#spark-cassandra-connector-driver_2.12;3.5.0 in central
	found org.scala-lang.modules#scala-collection-compat_2.12;2.11.0 in central
	found com.datastax.oss#java-driver-core-shaded;4.13.0 in central
	found com.datastax.oss#native-protocol;1.5.0 in central
	found com.datastax.oss#java-driver-shaded-guava;25.1-jre-graal-sub-1 in central
	found com.typesafe#config;1.4.1 in central
	found org.slf4j#slf4j-api;1.7.26 in central
	found io.dropwizard.metrics#metrics-core;4.1.18 in central
	found org.hdrhistogram#HdrHistogram;2.1.12 in central
	found org.reactivestreams#reactive-streams;1.0.3

✅ Spark Session successfully created!


In [ ]:

file_path = "../amazon_reviews.csv"

df_raw = spark.read.csv(
    file_path,
    header=True,
    inferSchema=True,
    sep=","
)

print(f"Total rows in file: {df_raw.count()}")
df_raw.printSchema() # Let's print the schema to verify column detection
df_raw.show(5, truncate=False)

In [8]:
spark.conf.set("spark.sql.legacy.timeParserPolicy", "LEGACY")

In [9]:
print(" Starting data transformation and load process for 'reviews_by_product'...")

# 1. Transform
df_reviews_by_product = df_raw.select(
    col("product_id").cast("string"),
    to_date(col("review_date"), "yyyy-MM-dd").alias("review_date"), # Convert string to proper DateType
    col("review_id").cast("string"),
    col("customer_id").cast("string"),
    col("star_rating").cast("int"),
    col("review_headline").cast("string"),
    col("review_body").cast("string")
).dropna(subset=["product_id", "review_date", "review_id"])
# IMPORTANT: Drop rows where any part of the Primary Key is null to prevent Cassandra insertion errors

# 2. Load
df_reviews_by_product.write \
    .format("org.apache.spark.sql.cassandra") \
    .mode("append") \
    .options(table="reviews_by_product", keyspace="amazon") \
    .save()

print("Data successfully loaded into the 'reviews_by_product' table!")

 Starting data transformation and load process for 'reviews_by_product'...


Data successfully loaded into the 'reviews_by_product' table!


In [10]:
print("Loading data into 'reviews_by_product_rating' and 'reviews_by_customer'...")

# Table 2: Reviews by product AND rating
df_reviews_by_product.write \
    .format("org.apache.spark.sql.cassandra") \
    .mode("append") \
    .options(table="reviews_by_product_rating", keyspace="amazon") \
    .save()
print("Success: 'reviews_by_product_rating' loaded!")

# Table 3: Reviews by customer (Notice we select only the columns needed for this table)
df_reviews_by_customer = df_reviews_by_product.select(
    "customer_id", "review_date", "review_id", "product_id", "star_rating", "review_headline"
)

df_reviews_by_customer.write \
    .format("org.apache.spark.sql.cassandra") \
    .mode("append") \
    .options(table="reviews_by_customer", keyspace="amazon") \
    .save()
print("Success: 'reviews_by_customer' loaded!")

⏳ Loading data into 'reviews_by_product_rating' and 'reviews_by_customer'...


✅ Success: 'reviews_by_product_rating' loaded!


✅ Success: 'reviews_by_customer' loaded!


In [ ]:
from pyspark.sql.functions import count, date_format

print("Preparing base dataframe for aggregations...")

#  Create a base dataframe for all data marts

df_base_marts = df_raw.select(
    col("product_id").cast("string"),
    col("customer_id").cast("string"),
    col("star_rating").cast("int"),
    col("verified_purchase").cast("string"),
    date_format(to_date(col("review_date"), "yyyy-MM-dd"), "yyyy-MM").alias("period")
).dropna(subset=["period"])

# Cache the dataframe so Spark doesn't read the CSV 4 times
df_base_marts.cache()


# Table 4: Top products by period

print("Calculating Table 4: top_products_by_period...")
df_top_products = df_base_marts.groupBy("period", "product_id") \
    .agg(count("*").alias("review_count"))

df_top_products.write \
    .format("org.apache.spark.sql.cassandra") \
    .mode("append") \
    .options(table="top_products_by_period", keyspace="amazon") \
    .save()
print("Success: Loaded data into 'top_products_by_period'")


# Table 5: Top customers (Verified Purchases)

print("Calculating Table 5: top_customers_verified...")
df_top_verified = df_base_marts.filter(col("verified_purchase") == "Y") \
    .groupBy("period", "customer_id") \
    .agg(count("*").alias("review_count"))

df_top_verified.write \
    .format("org.apache.spark.sql.cassandra") \
    .mode("append") \
    .options(table="top_customers_verified", keyspace="amazon") \
    .save()
print("Success: Loaded data into 'top_customers_verified'")


# Table 6: Top haters (1-2 stars)

print("Calculating Table 6: top_haters...")
df_top_haters = df_base_marts.filter(col("star_rating").isin(1, 2)) \
    .groupBy("period", "customer_id") \
    .agg(count("*").alias("review_count"))

df_top_haters.write \
    .format("org.apache.spark.sql.cassandra") \
    .mode("append") \
    .options(table="top_haters", keyspace="amazon") \
    .save()
print("Success: Loaded data into 'top_haters'")

# Table 7: Top backers (4-5 stars)

print("Calculating Table 7: top_backers...")
df_top_backers = df_base_marts.filter(col("star_rating").isin(4, 5)) \
    .groupBy("period", "customer_id") \
    .agg(count("*").alias("review_count"))

df_top_backers.write \
    .format("org.apache.spark.sql.cassandra") \
    .mode("append") \
    .options(table="top_backers", keyspace="amazon") \
    .save()
print("Success: Loaded data into 'top_backers'")


df_base_marts.unpersist()
print("ALL DATA MARTS SUCCESSFULLY LOADED!")